<a href="https://colab.research.google.com/github/PRAJEENS2024/prajeen-codeboosters-2026/blob/main/Day%203/Day_3_ELT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')
print(f'pandas version: {pd.__version__}')
print(f'requests version: {requests.__version__}')


All libraries imported successfully!
pandas version: 2.2.2
requests version: 2.32.4


In [ ]:
#====================================
#EXTRACT : Load raw messy sales data
#====================================
raw_df = pd.read_csv('messy_sales_data.csv')
print(f'Raw data loaded:{raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
print(f'Colmns:{raw_df.columns.tolist()}')
raw_df.head()

Raw data loaded:30 rows, 9 columns
Colmns:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
#====================================
#EXTRACT : Load raw messy sales data
#====================================

print('='*55)
print('DATA QUALITY DIAGNOSIS REPORT')
print('='*55)

#1.Missing Value
print('\n[1] MISSING VALUES per column: ')
print(raw_df.isnull().sum())

#2.Duplicates
print(f'\n[2] DUPLICATE ROWS : {raw_df.duplicated().sum()}')

#3. Data types
print('\n[3] DATA TYPES :')
print(raw_df.dtypes)

#4. Unique values in text columns (spot inconsistencie)
print('\n[4] UNIQUE CATEGORIES:', raw_df['category'].unique())
print('[4] Sample customer names:', raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_data values:', raw_df['order_date'].unique()[:6])

DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES per column: 
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS : 0

[3] DATA TYPES :
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_data values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
print('\n[1] MISSING VALUES IN QUANTITY COLUMN:')
print(raw_df['quantity'].isnull().sum())


[1] MISSING VALUES IN QUANTITY COLUMN:
3


In [ ]:
#====================================
#Create a working copy (ELT best practice)
#====================================
df = raw_df.copy()
print(f'Working copy created:{df.shape}')
print('raw_df is untouched ')

Working copy created:(30, 9)
raw_df is untouched 


In [ ]:
#====================================
#Fix #1: Handle Missing Values
#====================================
print('Before fixing nulls:', df.isnull().sum().sum(), 'total missing values')

df['customer_name'].fillna('Unknown Customer', inplace = True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace = True)
print(f'Filled missing quantity values with {median_qty}')

df['category'].fillna('Uncategorized', inplace = True)
print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')

Before fixing nulls: 7 total missing values
Filled missing quantity values with 2.0
After fixing nulls: 1 total missing values


In [ ]:
#====================================
#Fix #2: Remove Duplicates
#====================================
print(f'Before duplication :{len(df)} rows')
print(f'Duplicate rows:{df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name', 'product', 'order_id']])
df.drop_duplicates(inplace=True)
print(f'\nAfter deduplication: {len(df)} rows')
print(f'Rows removed: {len(raw_df) - len(df)}')

Before duplication :30 rows
Duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_id]
Index: []

After deduplication: 30 rows
Rows removed: 0


In [ ]:
print(df.isnull().sum())

order_id         0
customer_name    0
product          1
category         0
quantity         0
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64


In [ ]:
df['product'].fillna('NotListed', inplace = True)
print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')

After fixing nulls: 0 total missing values


In [ ]:
print('\nSample dates before parsing:')
print(df['order_date'].head(8).tolist())
df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst = False,
    errors = 'coerce'
)
nat_count = df['order_date'].isnull().sum()
print(f'\n')

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%B')
df['day'] = df['order_date'].dt.day_name()
print('\nSample dates after parsing:')
print(df[['order_date','year','month','day', 'month_name']].head(5))



Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']



Sample dates after parsing:
  order_date    year  month        day month_name
0 2024-01-05  2024.0    1.0     Friday    January
1 2024-01-07  2024.0    1.0     Sunday    January
2 2024-01-08  2024.0    1.0     Monday    January
3 2024-01-10  2024.0    1.0  Wednesday    January
4 2024-01-05  2024.0    1.0     Friday    January


In [ ]:
print('Before standardization:' , df['customer_name'].unique()[:6])
df['customer_name'] = (
    df['customer_name']
    .str.strip() # Remove leading/ trailing spaces
    .str.title() #
)
print('After standardization:',df['customer_name'].unique()[:6])

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']


In [ ]:
# fix data types +  create revenue column
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype(int)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['revenue'] = df['quantity'] * df['unit_price']
print("Revenue column created:")
print(df[['customer_name','product','quantity', 'unit_price', 'revenue']].head())
total_revenue = df['revenue'].sum()
print(f"\nTotal Revenue across all orders: ${total_revenue:.2f}")


Revenue column created:
  customer_name    product  quantity  unit_price  revenue
0  Ramesh Kumar     Laptop         2       45000    90000
1    Priya Nair  NotListed         1       15000    15000
2    Amit Verma   Keyboard         3        1200     3600
3  Sunita Patel    Monitor         2       22000    44000
4  Ramesh Kumar     Laptop         2       45000    90000

Total Revenue across all orders: $818000.00


In [ ]:
print('='*55)
print('POST-CLEANING VALIDATION REPORT')
print('='*55)
print(f'Original rows : {len(raw_df)}')
print(f'Cleaned rows : {len(df)}')
print(f'Rows removed: {len(raw_df) - len(df)} (duplicates)')
print(f'Total missing values: {df.isnull().sum().sum()}(should be 0)')
print(f'Total duplicates: {df.duplicated().sum()} (should be 0)')
print(f'Date nulls:{df["order_date"].isnull().sum()}')
print(f'Revenue NaN:{df["revenue"].isnull().sum()}')
print(f'Category:{df["category"].unique()}')
print('='*55)

all_clean =(
    df.isnull().sum().sum() == 0 and
    df.duplicated().sum() == 0
)
print(f'DATA IS CLEAN: {all_clean}')

POST-CLEANING VALIDATION REPORT
Original rows : 30
Cleaned rows : 30
Rows removed: 0 (duplicates)
Total missing values: 10(should be 0)
Total duplicates: 0 (should be 0)
Date nulls:2
Revenue NaN:0
Category:['Electronics' 'Accessories' 'Uncategorized']
DATA IS CLEAN: False
